# SNR of the validation recordings

One number per file: how far the annotated calls stand above the background of their own
recording. Aggregated per subset for the paper. Nothing else.

**Input:** `Development_Set/Validation_Set` (wav + sibling csv).
**Outputs:** two csv files, two LaTeX tables in `paper/tables/`, one figure in
`paper/figures/`, and a check figure.

---

## The measurement, in words

The dataset gives no clean recording of the noise, so SNR is estimated from the
annotations themselves. Split every recording's frames into two groups and compare them:

- **event frames** — the frame's centre falls inside an annotated **POS** event
- **background frames** — the frame is inside *no* annotation at all

Then, writing $P_{ev}$ for the mean power of the event frames and $P_{bg}$ for the
**median** power of the background frames:

$$\mathrm{SNR_{dB}} \;=\; 10\log_{10}\frac{P_{ev} - P_{bg}}{P_{bg}}$$

Three choices, each with a reason you can give a reviewer:

| choice | reason |
|---|---|
| a **±0.1 s guard band** around every annotation is in neither group | calls bleed slightly past their marked boundaries; without the guard that leakage lands in the noise estimate |
| **UNK** and **NEG** intervals are discarded entirely | they are neither confirmed call nor confirmed background |
| $P_{bg}$ is the **median**, not the mean | one passing car or one bout of another species would otherwise raise the noise floor of the whole recording |
| $P_{bg}$ is **subtracted** from $P_{ev}$ | the event frames contain the noise too; the numerator is the call's own power |

Power is short-time STFT power (1024-sample window, 256-sample hop, Hann) summed over
**50–11025 Hz** — the same framing and band as the feature front end in
`configs/train.yaml`, so this is the SNR *as the model sees it*. It is a ratio, so it is
unaffected by the per-file peak normalisation the pipeline applies.

## How to read the number

| SNR | meaning |
|---|---|
| **+20 dB** | call power is 100× the background |
| **+10 dB** | 10× |
| **0 dB** | the call's power equals the background's |
| **negative** | the call's power is *below* the background level — a faint call, or a narrowband call measured over a wide band |

Negative values are normal here and are explained in the last section, together with the
`flag` column, which marks the few recordings where the estimate breaks down.


## 0 — Configuration

In [ ]:
import os
from pathlib import Path

# configs/train.yaml -> path.eval_dir. Override without editing:  export DCASE_DATA_ROOT=...
DATA_ROOT = Path(os.environ.get("DCASE_DATA_ROOT", "/media/helios/Data/Uni/Masters/Dataset"))
VAL_DIR = DATA_ROOT / "Development_Set" / "Validation_Set"

# Front end -- identical to configs/train.yaml -> features and data_preparation.py
N_FFT, HOP = 1024, 256
FMIN, FMAX = 50, 11025

GUARD_S = 0.10          # seconds excluded either side of every annotation
MIN_BG_FRAMES = 100     # below this there is no usable noise estimate
SNR_FLOOR_DB = -30.0    # floor for events measured at or below the background
BLOCK_FRAMES = 1024     # streaming block size, in frames

# dB to 0.1 is the meaningful precision; finer would be false precision.
N_DECIMALS = 1

# Your per-subset result tables (paper/tables/per_subset_*.tex) show five subsets, because
# PB and PB24 merge when notebooks/file_to_subset.json is absent -- both hold BUK* files.
# Set MERGE_SUBSETS = {"PB24": "PB"} if you want this table to line up row for row.
MERGE_SUBSETS = {}

OUT_DIR = Path("snr")
PAPER_TABLES_DIR = Path("../../../paper/tables").resolve()
PAPER_FIGURES_DIR = Path("../../../paper/figures").resolve()
for d in (OUT_DIR, PAPER_TABLES_DIR, PAPER_FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"validation set : {VAL_DIR}\n                 exists: {VAL_DIR.is_dir()}")
print(f"tables         : {PAPER_TABLES_DIR}")
print(f"figures        : {PAPER_FIGURES_DIR}")

## 1 — Helpers

Audio is streamed rather than loaded whole: a full-file spectrogram of an hour-long
recording at this hop would be over a gigabyte.

In [ ]:
import inspect
import traceback
import warnings

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

EPS = 1e-20
META_COLS = ["Audiofilename", "Starttime", "Endtime"]

# librosa.stream() only grew an `sr` argument in 1.0, and in 1.1 its default becomes 22050,
# i.e. it starts resampling. Frame indices here come from sf.info().frames at the file's
# native rate, so pin sr=None where the argument exists.
try:
    _STREAM_SR = {"sr": None} if "sr" in inspect.signature(librosa.stream).parameters else {}
except (TypeError, ValueError):     # pragma: no cover
    _STREAM_SR = {}
print(f"librosa {librosa.__version__}"
      f"{' (native rate pinned)' if _STREAM_SR else ' (reads native rate)'}")


def read_annotations(csv_path):
    """Return (pos, other): [start, end] arrays in seconds.

    Label columns are everything that is not a META_COL, so this works for the single-'Q'
    validation csvs and for the per-class training ones alike (the pattern used in
    single_file_analysis.ipynb). A row counts as POS if any label column says POS;
    everything else annotated (UNK, NEG) goes to `other` and is excluded from both groups.
    """
    df = pd.read_csv(csv_path, header=0, index_col=False)
    df.columns = [str(c).strip() for c in df.columns]
    class_cols = [c for c in df.columns if c not in META_COLS]
    labels = df[class_cols].astype(str).apply(lambda s: s.str.strip().str.upper())
    is_pos = (labels == "POS").any(axis=1).to_numpy()
    iv = df[["Starttime", "Endtime"]].astype(float).to_numpy()
    return iv[is_pos], iv[~is_pos]


def frame_labels(t, pos, other):
    """0 = background, 1 = event, 2 = excluded (guard band, UNK/NEG)."""
    lab = np.zeros(len(t), np.int8)
    annotated = [iv for iv in (pos, other) if len(iv)]
    for s, e in (np.vstack(annotated) if annotated else np.empty((0, 2))):
        lab[(t >= s - GUARD_S) & (t <= e + GUARD_S)] = 2   # guard everything,
    for s, e in pos:
        lab[(t >= s) & (t <= e)] = 1                       # then carve the events back out
    return lab


def stream_power(wav_path):
    """Yield (bins, frames) power blocks at the file's native sample rate."""
    blocks = librosa.stream(str(wav_path), block_length=BLOCK_FRAMES, frame_length=N_FFT,
                            hop_length=HOP, mono=True, dtype=np.float32, **_STREAM_SR)
    for y in blocks:
        if len(y) < N_FFT:          # ragged tail: no complete frame left
            break
        # center=False is what keeps streamed blocks aligned frame-for-frame
        yield np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP, center=False)) ** 2


def measure(wav_path, csv_path, keep_trace=False):
    """One streaming pass over a file -> its SNR row (and optionally the power trace)."""
    info = sf.info(str(wav_path))
    sr = info.samplerate
    n_frames = 1 + max(0, info.frames - N_FFT) // HOP

    pos, other = read_annotations(csv_path)
    t = (np.arange(n_frames) * HOP + N_FFT / 2) / sr        # frame centre times
    lab = frame_labels(t, pos, other)

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    band = (freqs >= FMIN) & (freqs <= min(FMAX, sr / 2))

    power = np.zeros(n_frames)
    sum_ev, sum_bg = np.zeros(int(band.sum())), np.zeros(int(band.sum()))
    n_ev = n_bg = i = 0
    for S in stream_power(wav_path):
        n = min(S.shape[1], n_frames - i)
        if n <= 0:
            break
        S = S[band, :n]
        power[i:i + n] = S.sum(axis=0)
        block = lab[i:i + n]
        ev, bg = block == 1, block == 0
        if ev.any():
            sum_ev += S[:, ev].sum(axis=1)
            n_ev += int(ev.sum())
        if bg.any():
            sum_bg += S[:, bg].sum(axis=1)
            n_bg += int(bg.sum())
        i += n
    power, lab, t = power[:i], lab[:i], t[:i]      # frames the stream never reached

    p_ev = power[lab == 1].mean() if (lab == 1).any() else np.nan
    p_bg = np.median(power[lab == 0]) if n_bg >= MIN_BG_FRAMES else np.nan

    # 10*log10((p_ev - p_bg) / p_bg), with the two ways it can degenerate made explicit
    flag = ""
    if not np.isfinite(p_ev):
        snr_db, flag = np.nan, "no POS events"
    elif not np.isfinite(p_bg):
        snr_db, flag = np.nan, f"< {MIN_BG_FRAMES} background frames"
    elif p_ev <= p_bg:
        snr_db, flag = SNR_FLOOR_DB, "events at/below background"
    else:
        snr_db = 10 * np.log10((p_ev - p_bg) / p_bg)

    row = {
        "filename": wav_path.name,
        "sr": sr,
        "duration_s": round(info.duration, 1),
        "n_events": len(pos),
        "event_dur_median_s": round(float(np.median(pos[:, 1] - pos[:, 0])), 3) if len(pos) else np.nan,
        "event_frac": round(float((lab == 1).mean()), 4) if len(lab) else np.nan,
        "bg_frac": round(float((lab == 0).mean()), 4) if len(lab) else np.nan,
        "n_bg_frames": n_bg,
        "p_ev_db": 10 * np.log10(max(p_ev, EPS)) if np.isfinite(p_ev) else np.nan,
        "p_bg_db": 10 * np.log10(max(p_bg, EPS)) if np.isfinite(p_bg) else np.nan,
        "snr_db": snr_db,
        "flag": flag,
    }
    trace = {"t": t, "power": power, "lab": lab, "pos": pos} if keep_trace else None
    return row, trace

## 2 — Measure every validation file

The **subset** is the folder the wav sits in (`.../Validation_Set/ME/ME1.wav` → `ME`),
which is how the DCASE evaluator derives subsets, so these labels are the same ones your
result tables use.

In [ ]:
tasks, orphans = [], []
for wav in sorted(VAL_DIR.rglob("*.wav")):
    csv = wav.with_suffix(".csv")
    if csv.exists():
        subset = MERGE_SUBSETS.get(wav.parent.name, wav.parent.name)
        tasks.append((subset, wav, csv))
    else:
        orphans.append(wav)

if not tasks:
    raise RuntimeError(f"No wav/csv pairs under {VAL_DIR} -- check DATA_ROOT.")
if orphans:
    print(f"{len(orphans)} wav(s) have no csv and were skipped:")
    for wav in orphans:
        print(f"   {wav}")

rows, failed = [], []
for subset, wav, csv in tqdm(tasks, desc="files"):
    try:
        row, _ = measure(wav, csv)
    except Exception as exc:
        failed.append((wav.name, f"{type(exc).__name__}: {exc}", traceback.format_exc()))
        continue
    rows.append({"subset": subset, **row, "path": str(wav)})

# Failures are reported before the table is built: if they all fail, `rows` is empty and a
# column access would raise something unrelated to the real cause.
if failed:
    print(f"\n{len(failed)} of {len(tasks)} file(s) failed:")
    for name, err, _ in failed[:10]:
        print(f"   {name}: {err}")
    print(f"\nFull traceback for the first failure ({failed[0][0]}):\n")
    print(failed[0][2])
if not rows:
    raise RuntimeError(f"All {len(tasks)} file(s) failed -- see the traceback above.")

per_file = (pd.DataFrame(rows)
            .sort_values(["subset", "snr_db"], ascending=[True, False], ignore_index=True))
per_file.to_csv(OUT_DIR / "validation_snr_per_file.csv", index=False)

print(f"\n{len(per_file)} files in {per_file['subset'].nunique()} subsets "
      f"-> {OUT_DIR / 'validation_snr_per_file.csv'}")
print(f"median SNR {per_file['snr_db'].median():.1f} dB, "
      f"range {per_file['snr_db'].min():.1f} to {per_file['snr_db'].max():.1f} dB")
flagged = per_file[per_file["flag"] != ""]
if len(flagged):
    print(f"\n{len(flagged)} file(s) flagged -- see the last section:")
    for r in flagged.itertuples():
        print(f"   {r.filename:<40} {r.flag}")
dupes = per_file.loc[per_file.duplicated("filename", keep=False), "filename"].unique()
if len(dupes):
    print(f"\nNote: {list(dupes)} appears in more than one subset, so it has one row per "
          "copy. Correct here -- each copy is a different recording.")

## 3 — Table 1: per file

Sorted loudest first within each subset. `paper/tables/snr_per_file.tex` is the same table
as a standalone LaTeX block; the csv written above carries the intermediate quantities
(`p_ev_db`, `p_bg_db`, frame counts) if a reviewer asks how a number arose.

In [ ]:
def save_tex(df, filename, caption, label, **to_latex_kwargs):
    """Write `df` as a standalone LaTeX table -- same helper as the other paper notebooks."""
    kwargs = dict(float_format=f"%.{N_DECIMALS}f", na_rep="—", caption=caption, label=label)
    kwargs.update(to_latex_kwargs)
    out = PAPER_TABLES_DIR / filename
    out.write_text(df.to_latex(**kwargs))
    print(f"  saved {out}")


table_file = (per_file[["subset", "filename", "duration_s", "n_events", "snr_db"]]
              .rename(columns={"subset": "Subset", "filename": "File",
                               "duration_s": "Duration (s)", "n_events": "Events",
                               "snr_db": "SNR (dB)"}))

save_tex(table_file, "snr_per_file.tex",
         caption=("Estimated signal-to-noise ratio of each recording in the DCASE 2024 "
                  "Task~5 validation set. SNR is the ratio of the mean power of annotated "
                  "positive-event frames, minus the noise floor, to the noise floor itself, "
                  "where the floor is the median power of the frames outside every "
                  "annotation (100~ms guard band excluded). Power is summed over "
                  "50--11025~Hz on 1024-sample frames with 256-sample hop."),
         label="tab:snr_per_file", index=False)

table_file.round(N_DECIMALS)

## 4 — Table 2: per subset

The one for the paper body. **Median** rather than mean, because with a handful of files
per subset a single unusual recording would move a mean a long way; the min and max columns
show the spread that median hides.

In [ ]:
by_subset = (per_file.groupby("subset")
             .agg(Files=("filename", "size"),
                  Events=("n_events", "sum"),
                  Hours=("duration_s", lambda s: s.sum() / 3600),
                  Median=("snr_db", "median"),
                  Min=("snr_db", "min"),
                  Max=("snr_db", "max"))
             .sort_values("Median"))
by_subset.index.name = "Subset"
by_subset.columns = ["Files", "Events", "Hours", "Median SNR (dB)", "Min (dB)", "Max (dB)"]

overall = pd.DataFrame([{
    "Files": len(per_file),
    "Events": int(per_file["n_events"].sum()),
    "Hours": per_file["duration_s"].sum() / 3600,
    "Median SNR (dB)": per_file["snr_db"].median(),
    "Min (dB)": per_file["snr_db"].min(),
    "Max (dB)": per_file["snr_db"].max(),
}], index=["All"])
overall.index.name = "Subset"

by_subset_out = pd.concat([by_subset, overall])
by_subset_out.to_csv(OUT_DIR / "validation_snr_by_subset.csv")

save_tex(by_subset_out, "snr_by_subset.tex",
         caption=("Estimated signal-to-noise ratio per subset of the DCASE 2024 Task~5 "
                  "validation set. Values are per-recording SNR in dB, summarised by the "
                  "median across the recordings of each subset; Min and Max give the range. "
                  "Subsets are ordered from noisiest to cleanest. See "
                  "Table~\\ref{tab:snr_per_file} for the individual recordings."),
         label="tab:snr_by_subset")

by_subset_out.round(N_DECIMALS)

## 5 — Figure: every file, grouped by subset

One dot per recording, subsets ordered noisiest to cleanest, with each subset's median
marked. This is the figure to put next to the per-subset table — it shows *how much the
recordings within a subset differ*, which the median alone hides.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.alpha": 0.4, "font.size": 10,
})
BLUE, RED, INK, GREY = "#2a78d6", "#d03b3b", "#3a3a3a", "#9a9a9a"


def save_fig(fig, stem, paper=True):
    fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", dpi=150)
    print(f"  saved {OUT_DIR / f'{stem}.png'}")
    if paper:
        fig.savefig(PAPER_FIGURES_DIR / f"{stem}.pdf", bbox_inches="tight")
        print(f"  saved {PAPER_FIGURES_DIR / f'{stem}.pdf'}")


order = list(by_subset.index)
usable = per_file.dropna(subset=["snr_db"])

fig, ax = plt.subplots(figsize=(1.15 * len(order) + 2.2, 4.0))
rng = np.random.default_rng(1234)
for i, subset in enumerate(order):
    g = usable[usable["subset"] == subset]
    clean, odd = g[g["flag"] == ""], g[g["flag"] != ""]
    for sub, colour, label in ((clean, BLUE, "recording"),
                               (odd, RED, "estimate breaks down (see notes)")):
        if len(sub):
            ax.scatter(i + rng.uniform(-0.14, 0.14, len(sub)), sub["snr_db"], s=34,
                       color=colour, alpha=0.85, edgecolor="white", linewidth=0.6,
                       zorder=3, label=label)
    med = g["snr_db"].median()
    ax.plot([i - 0.3, i + 0.3], [med, med], color=INK, lw=2, zorder=4)
    ax.annotate(f"{med:.1f}", (i + 0.33, med), fontsize=9, color=INK, va="center")

ax.axhline(0, color=GREY, lw=1, ls=(0, (4, 3)), zorder=0)
ax.annotate("0 dB: call power = background power", (len(order) - 0.5, 0),
            textcoords="offset points", xytext=(0, 5), ha="right", fontsize=8, color=GREY)
ax.set_xticks(range(len(order)),
              [f"{s}\n(n={int(by_subset.loc[s, 'Files'])})" for s in order])
ax.set_xlim(-0.6, len(order) - 0.2)
ax.set_xlabel("Subset (noisiest to cleanest)")
ax.set_ylabel("Estimated SNR (dB)")

handles, labels = ax.get_legend_handles_labels()
seen = dict(zip(labels, handles))
if len(seen) > 1:
    ax.legend(seen.values(), seen.keys(), frameon=False, fontsize=9, loc="upper left")

save_fig(fig, "snr_by_subset")
plt.show()

## 6 — Check figure: is the estimate measuring the calls?

Not for the paper body, but this is what to show if the estimator is questioned — and what
to look at yourself before quoting any number. For the loudest and the quietest recording:

- **left** — 30 s of frame power, annotated events shaded, the noise floor drawn across.
  The shaded stretches should sit above the line, and by roughly the SNR in the title.
- **right** — the average spectrum of event frames against background frames. They should
  separate in the band where the species actually calls.

If the shading does not line up with the peaks, the annotation parse is wrong and every
number above is wrong with it.

In [ ]:
def check_figure(filename, window_s=30):
    row = per_file[per_file["filename"] == filename].iloc[0]
    wav = Path(row["path"])
    _, trace = measure(wav, wav.with_suffix(".csv"), keep_trace=True)
    t, power, lab, pos = trace["t"], trace["power"], trace["lab"], trace["pos"]
    bg = np.median(power[lab == 0])

    t0 = max(0.0, pos[0][0] - 2) if len(pos) else 0.0
    win = (t >= t0) & (t <= t0 + window_s)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
    ax1.plot(t[win], 10 * np.log10(np.maximum(power[win], EPS)), lw=0.7, color=INK)
    for s, e in pos:
        if e > t0 and s < t0 + window_s:
            ax1.axvspan(s, e, color=BLUE, alpha=0.18, lw=0)
    ax1.axhline(10 * np.log10(max(bg, EPS)), color=RED, lw=1.4,
                label="noise floor $P_{bg}$")
    ax1.set_xlim(t0, t0 + window_s)
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Frame power (dB, arbitrary ref.)")
    ax1.set_title(f"{row['subset']} · {filename[:34]} · SNR {row['snr_db']:.1f} dB",
                  loc="left", fontsize=10)
    ax1.legend(frameon=False, fontsize=9)

    freqs = librosa.fft_frequencies(sr=row["sr"], n_fft=N_FFT)
    band = (freqs >= FMIN) & (freqs <= min(FMAX, row["sr"] / 2))
    # recomputed here only for the plot -- cheap next to the streaming pass
    S_ev, S_bg = np.zeros(int(band.sum())), np.zeros(int(band.sum()))
    n_ev = n_bg = i = 0
    for S in stream_power(wav):
        n = min(S.shape[1], len(lab) - i)
        if n <= 0:
            break
        S = S[band, :n]
        blk = lab[i:i + n]
        if (blk == 1).any():
            S_ev += S[:, blk == 1].sum(axis=1); n_ev += int((blk == 1).sum())
        if (blk == 0).any():
            S_bg += S[:, blk == 0].sum(axis=1); n_bg += int((blk == 0).sum())
        i += n
    ax2.plot(freqs[band], 10 * np.log10(np.maximum(S_ev / max(n_ev, 1), EPS)), lw=1.8,
             color=BLUE, label="event frames")
    ax2.plot(freqs[band], 10 * np.log10(np.maximum(S_bg / max(n_bg, 1), EPS)), lw=1.8,
             color=RED, label="background frames")
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Mean power (dB, arbitrary ref.)")
    ax2.set_title(f"{row['n_events']} events, median "
                  f"{row['event_dur_median_s']:.2f} s", loc="left", fontsize=10)
    ax2.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    return fig


for tag, fname in [("loudest", usable.loc[usable["snr_db"].idxmax(), "filename"]),
                   ("quietest", usable.loc[usable["snr_db"].idxmin(), "filename"])]:
    save_fig(check_figure(fname), f"snr_check_{tag}", paper=False)
    plt.show()

## Writing it up

A paragraph to adapt for the methods or dataset section:

> Because the dataset provides no isolated recording of the background, we estimate each
> recording's signal-to-noise ratio from its reference annotations. Short-time power is
> computed on 1024-sample Hann-windowed frames with a 256-sample hop and summed over
> 50--11025~Hz, matching the feature front end. Frames whose centre lies within an
> annotated positive event form the event set; frames lying outside every annotation, with
> a 100~ms guard band excluded on each side, form the background set, and UNK and NEG
> intervals are discarded. Writing $P_{ev}$ for the mean power of the event frames and
> $P_{bg}$ for the median power of the background frames, the estimate is
> $\mathrm{SNR} = 10\log_{10}\!\big((P_{ev}-P_{bg})/P_{bg}\big)$~dB. The median is used for
> the background so that sporadic non-target sounds do not inflate the noise estimate, and
> $P_{bg}$ is subtracted from $P_{ev}$ because the event frames contain the background as
> well as the call.

**State these limits too** — each is a question a reviewer can ask:

1. **The floor is a noise floor, not silence.** Un-annotated stretches contain wind, rain,
   other species. That lowers the reported SNR, and arguably it should: it is the
   background a detector actually competes with. It does mean the values are not comparable
   to a laboratory SNR.
2. **The estimate is only as complete as the annotations.** A target call that was never
   annotated is counted as background, which biases that recording's SNR downwards.
3. **It is broadband.** Summing 50--11025~Hz dilutes a narrowband call across bins that
   hold only noise, so a clearly audible narrowband species reads lower than an
   equally audible broadband one. Compare *within* a subset freely; across subsets,
   remember the species differ in bandwidth.
4. **Events are bouts, not syllables.** Where an annotation spans a call sequence with gaps,
   the quiet gaps are averaged into $P_{ev}$ and pull the value down.

**Negative values are expected** and worth one sentence in the text. The measure turns
negative once the call's power above the floor is smaller than the floor itself, so a
recording whose annotated regions are only ~2~dB above their background lands near
$-4$~dB. It is a statement about the recording, not a computational artefact.

**The `flag` column** marks the cases where the estimate genuinely fails, and they should
be excluded from any claim rather than reported as very low SNR:

| flag | what happened | what to do |
|---|---|---|
| *events at/below background* | $P_{ev} \le P_{bg}$, so the ratio is undefined; the value is clamped to the floor | inspect with `check_figure(name)`; usually a long bout annotation that is mostly silence, or a background louder than the target |
| *< 100 background frames* | almost the whole recording is annotated, leaving nothing to measure the floor from | report as not estimable |
| *no POS events* | no positive annotation in the file | excluded by construction |

**Reproducing the numbers:** `snr/validation_snr_per_file.csv` keeps $P_{ev}$, $P_{bg}$ and
the frame counts behind every row, so any single value can be traced back to the file.
